In [0]:
!pip install shap

###  Respondent Modelling

Workbook reports on the following:

- **Random Forest** for non-linear feature importance
- **SHAP values** for interpretable feature contributions
- **OLS coefficients** for linear effect magnitude comparison

## Key Metrics Explained

### SHAP Feature Importance
SHAP (SHapley Additive exPlanations) values measure each feature's contribution to individual predictions. The **mean absolute SHAP value** represents how much, on average, a feature moves the prediction away from the baseline. Higher values = more influential features. Unlike RF importance, SHAP accounts for feature interactions and provides consistent, theoretically-grounded importance scores.

### SHAP Contribution (Direction)
The **mean SHAP value** (signed) shows whether a feature typically *increases* or *decreases* wonkiness:
- **Positive contribution**: Higher/present values of this feature increase wonky behavior
- **Negative contribution**: Higher/present values of this feature decrease wonky behavior

### OLS Coefficient
The standardized OLS coefficient represents the **linear marginal effect** - how much the outcome changes (in standard deviations) for a one standard deviation change in the feature, holding others constant. Useful for understanding effect magnitude in interpretable units.

### Feature Interactions
SHAP interaction values capture how pairs of features jointly affect predictions beyond their individual effects. Strong interactions indicate features that work together (or against each other) in complex ways - e.g., the effect of `is_evening` might depend on `is_friday`.


### Methods Note

**Model:** Random Forest Regressor with GroupKFold CV (clustered by user)
**Interpretation:** SHAP TreeExplainer values

### Why RF for a binary outcome?** 
We treat wonkiness as a continuous probability (0-1 scale), allowing us to detect nuanced effects. 
SHAP values can be interpreted as percentage point changes in wonkiness probability.

###Caveats:###
- High VIF features (e.g., `is_weekend` ↔ `is_saturday`) may have unstable individual estimates
- Interactions are based on a 250-observation sample — directional guidance only


#### Setup & Data Load

In [0]:
# Notebook Settings
import sys
import yaml
import shap
import warnings

warnings.filterwarnings("ignore")
import mlflow

mlflow.autolog(disable=True)

# Core imports
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import os

sys.path.append(os.path.join(os.path.dirname(os.getcwd()), "src"))

# Local modules
from modelling.modelling import (
    build_random_forest,
    compute_shap_values,
    compute_shap_interactions,
    get_shap_contributions,
    create_feature_summary,
)
from modelling.modelling_utils import calculate_vif, extract_stat_coefficients
from modelling.modelling_visualization import (
    plot_feature_comparison,
    plot_shap_contributions,
    plot_interactions,
)

In [0]:
# Load configs
with open('../configs/models.yaml', 'r') as f:
    model_config = yaml.safe_load(f)

with open('../configs/data_paths.yaml', 'r') as f:
    paths_config = yaml.safe_load(f)

pd.set_option('display.max_columns', None)

In [0]:
# Loading user level dataframe
notebook_path = os.getcwd()
repo_root = os.path.abspath(os.path.join(notebook_path, ".."))
misc_dir = os.path.join(repo_root, "misc")

user_df_input_path = os.path.join(misc_dir,
                           os.path.basename(paths_config['output_files']['user_info_df_post_eda']))

test_results_df_input_path = os.path.join(misc_dir,
                           os.path.basename(paths_config['output_files']['test_results_df']))

user_info_df = pd.read_parquet(user_df_input_path)
test_results_df = pd.read_csv(test_results_df_input_path)

user_info_df = user_info_df[~user_info_df['wonky_study_count'].isna()]
print(f"Data loaded: {user_info_df.shape}")

In [0]:
user_info_df = user_info_df[~user_info_df['survey_type'].isna()]

In [0]:
sorted(test_results_df[test_results_df['significant_both']]['feature'])

In [0]:
# CONTROLL_SETTING = None
# CONTROLL_SETTING = 'control'
CONTROLL_SETTING = 'exposed'

baseline wonkiness of dataset is 42.4%

binary outcome, RF model predicts on a 0-1 proba scale and SHAPS should be percentage points


#### Feature Selection & VIF Analaysis (Check for highly colinear features)

In [0]:
CONTROLL_SETTING

In [0]:
feature_cols = test_results_df[test_results_df['significant_both']]['feature'].tolist()

vif_results = calculate_vif(
    user_info_df, 
    test_results_df,
    feature_cols, 
    threshold_high=model_config['vif_analysis']['threshold_high'],
    threshold_moderate=model_config['vif_analysis']['threshold_moderate'],
)

display(vif_results.head(20))

In [0]:
if CONTROLL_SETTING:
  controll_setting = CONTROLL_SETTING
  user_info_df = user_info_df[user_info_df[f'survey_type_{controll_setting}'] == 1].reset_index(drop=True)

In [0]:
user_info_df['survey_type'].unique()

In [0]:
user_info_df

In [0]:
# Manual feature adjustments

""" Remove features you want to exclude typically if two features have infinity VIF within the same feature_set remove 1 one of them. For example if the featureset is operating system [ditr_os_ios, ditr_os_android] remove one of them. """

if CONTROLL_SETTING == 'control':
    opposite_feature = "survey_type_exposed"
elif CONTROLL_SETTING == 'exposed':
    opposite_feature = "survey_type_control"
else:
    opposite_feature = ""
    

features_to_remove = [
    "ditr_os_android",
    "exposure_band_control",
    "exposure_band_exposed",
    "is_weekend",
    "is_very_fast",
    "is_afternoon",
    "weeks_active_before_task",
    "ditr_os_ios",
    "manufacturer_category_apple",
    "share_location_data_False",
    # "days_active_before_task",
] + [opposite_feature] 

# Add features you want to include
features_to_add = [
]

# Final feature list
feature_cols = [f for f in feature_cols if f not in features_to_remove]
feature_cols.extend([f for f in features_to_add if f not in feature_cols])

# Validate features exist in data
available_features = [f for f in feature_cols if f in user_info_df.columns]
missing_features = [f for f in feature_cols if f not in user_info_df.columns]

if missing_features:
    print(f"Missing features (removed): {missing_features}")

feature_cols = available_features
print(f"\nFinal feature count: {len(feature_cols)}")

In [0]:
# check lower for VIFS

feature_cols2 = [i for i in feature_cols if i not in features_to_remove]

vif_results2 = calculate_vif(
    user_info_df, 
    test_results_df,
    feature_cols2, 
    threshold_high=model_config['vif_analysis']['threshold_high'],
    threshold_moderate=model_config['vif_analysis']['threshold_moderate'],
)

display(vif_results2.head(20))

#### Random Forest Model

In [0]:
user_info_df[feature_cols].sum().round(2)

In [0]:
mlflow.autolog(disable=True)

In [0]:
rf_config = model_config['random_forest']['default']
gs_config = model_config['random_forest']['gridsearch']

rf_model, X_data, cv_metrics, rf_importance = build_random_forest(
    df=user_info_df,
    feature_cols=feature_cols,
    outcome_var=model_config['target']['outcome_var'],
    user_id_var=model_config['cross_validation']['user_id_var'],
    n_estimators=rf_config['n_estimators'],
    max_depth=rf_config['max_depth'],
    max_features=rf_config['max_features'],
    n_splits=model_config['cross_validation']['n_splits'],
    do_gridsearch=gs_config['enabled'],
    param_grid=gs_config['param_grid'] if gs_config['enabled'] else None,
)

Used all the data and got much higher R sqaured >> should improve as more samples come in just not powerful enough yet

In [0]:
mlflow.autolog(disable=True)

#### SHAP Analysis

In [0]:
shap_config = model_config['shap']

explainer, shap_values, shap_importance = compute_shap_values(
    model=rf_model,
    X=X_data,
    sample_size=shap_config['sample_size'],
)

In [0]:
# SHAP contributions (direction of effect)
shap_contributions = get_shap_contributions(shap_values, X_data)
display(shap_contributions.head(50))


contribution interpretation if baseline wonkyness is 0.42 or 42% and days_active_before_task has a mean_contribution of +0.004119 (+0.41pp) it's absolute effect is + +0.41 percentage points and it's relative effect is 0.0041/0.42 = 0.0976 (+9.76%)


In [0]:
shap_contributions['mean_contribution_pp_change'] = (shap_contributions['mean_contribution'] * 100).round(2)
display(shap_contributions.head(50))

days active: most protective feature is as an addiitonal day results in ~ -0.6pp chance of wonky ness 

mean_contribution >> Average shap value across all observations.


abs_contribution >> Average absolute shap value across all observations.

In [0]:
# # Feature interactions (computationally expensive)
# interaction_values, interaction_df = compute_shap_interactions(
#     explainer=explainer,
#     X=X_data,
#     sample_size=shap_config['interaction_sample_size'],
#     top_n=shap_config['max_display'],
# )

In [0]:
import pickle

# pickle.dump(interaction_values, open("interaction_values.pkl", "wb"))
# interaction_df.to_csv("interaction_df.csv")

interaction_values = pickle.load(open("interaction_values.pkl", "rb"))
interaction_df = pd.read_csv("interaction_df.csv")

In [0]:
# X_sample = X_data.sample(n=1000, random_state=42)

# # Now the plot will work
# shap.dependence_plot(
#     "days_active_before_task", 
#     shap_values, 
#     X_sample,
#     interaction_index="days_active_201_to_250"
# )

Average incremental synergy of both

Days active being highly synergistic suggests the effect of exposure, tasks done etc depends alot of user tenure.

Consider tenure adjustment to any existing or planned processes to reduce wonkieness

#### Extract OLS coefs from testing results

In [0]:
from modelling.modelling_utils import calculate_vif, extract_stat_coefficients

In [0]:
# Extract OLS coefficients from test_results_df
stat_coefficients = extract_stat_coefficients(
    test_results_df=test_results_df,
    feature_cols=feature_cols,
)

display(stat_coefficients.head(20))

#### Combine all for Results Table

In [0]:
stat_coefficients

In [0]:
# Create unified summary
feature_summary = create_feature_summary(
    rf_importance=rf_importance,
    shap_importance=shap_importance,
    shap_contributions=shap_contributions,
    stats_coefficients=stat_coefficients,
    vif_data=vif_results,
)

# Feature summary tweaks
baseline_wonky_rate = user_info_df["wonky_study_count"].mean()
print(f"Baseline wonkiness rate: {baseline_wonky_rate:.2%}")
print(f"(This is the average probability of wonky behavior across all respondents)")

feature_summary["effect_pp"] = feature_summary["mean_contribution"] * 100  # percentage points
feature_summary["effect_relative_pct"] = (feature_summary["mean_contribution"] / baseline_wonky_rate * 100).round(3)

feature_summary["shap_importance"] = feature_summary["shap_importance"].round(3)

# Display key columns
display_cols = [
    "rank",
    "feature",
    "shap_importance",
    "effect_pp",
    "effect_relative_pct",
    "direction",
    "ols_interpretation_short",
    "lr_interpretation_short",
]
display(feature_summary[display_cols].head(30))

In [0]:
feature_summary['same_direction_signal'] = (
    (np.sign(feature_summary['mean_contribution']) == np.sign(feature_summary['ols_coefficient'])) & 
    (np.sign(feature_summary['mean_contribution']) == np.sign(feature_summary['odds_ratio'] - 1))
)

In [0]:
feature_summary

#### Visualizations

In [0]:
from modelling.modelling_visualization import (
    plot_feature_comparison,
    plot_shap_contributions,
    plot_interactions,
)

shap & rf importance pct is basically a share of prediction influence for example 32 means 32% of models decision making of wonky vs non wonky.

rf - how consistent results remain when feature is in tree
shap - how much a feature moves prediction from baseline (ave number of wonky studies)

shap tends to be more advanced and accurate in terms of importance but will often agree

In [0]:
# Feature comparison: SHAP importance vs RF importance
fig_comparison = plot_feature_comparison(
    feature_summary,
    top_n=10,
    data_cols=["shap_importance_pct", "effect_relative_pct"],
    label_cols=["shap_importance_pct", "effect_relative_pct"],   
)
fig_comparison.show()

In [0]:
# Feature comparison: SHAP importance vs RF importance
fig_comparison = plot_feature_comparison(
    feature_summary,
    top_n=10,
    data_cols=["shap_importance_pct", "rf_importance_pct"],
    label_cols=["shap_importance_pct", "rf_importance_pct"],
)
fig_comparison.show()

note log odds tend to provide more extreme results on the higher ends
for extreme numbers like 8.52x check sample size and balance of wonky and non wonkys obs -> tends to flare up when samples are low.

In [0]:
feature_summary['odds_ratio2'] = feature_summary['odds_ratio'] - 1 # tweak due to odds ratio interpretation

# Feature comparison: SHAP importance vs OLS coefficient
fig_comparison = plot_feature_comparison(
    feature_summary,
    top_n=10,
    data_cols=["odds_ratio2", "ols_coefficient"],
    label_cols=["lr_interpretation_short", "ols_interpretation_short"],
)
fig_comparison.show()

In [0]:
# Feature comparison: SHAP importance vs OLS coefficient

feature_summary['odds_ratio2'] = feature_summary['odds_ratio'] - 1 # tweak due to odds ratio interpretation

fig_comparison = plot_feature_comparison(
    feature_summary,
    top_n=10,
    data_cols=["odds_ratio2", "effect_relative_pct"],
    label_cols=["lr_interpretation_short", "effect_relative_pct"],
)
fig_comparison.show()

In [0]:
# Feature comparison: SHAP importance vs OLS coefficient

fig_comparison = plot_feature_comparison(
    feature_summary,
    top_n=10,
    data_cols=["ols_coefficient", "effect_relative_pct"],
    label_cols=["ols_interpretation_short", "effect_relative_pct"],
)
fig_comparison.show()

What you want to look for here is where both point in same direction

OLS - is a result of statistical test (stat sig reads)
mean_contribution - is a result of a attribution/ranking algo (SHAPLEY VALUES)

May not necessarily say the same thing since ols is just a single variable test whilst SHAPs estimates are more interaction dependant. 

Wednesday, 4pm _ consistent wonky driver

days_active_before_task, taskTitle_Awesome advertising task! _ consistent anti wonky driver

In [0]:
feature_summary

In [0]:
# SHAP contributions (direction of effect)
fig_contributions = plot_shap_contributions(feature_summary, top_n=len(feature_summary))
fig_contributions.show()

In [0]:
# SHAP contributions (direction of effect)
fig_contributions = plot_shap_contributions(feature_summary, top_n=20)
fig_contributions.show()

TODO chart task_completed_PWABTC

In [0]:
# Feature interactions
fig_interactions = plot_interactions(interaction_df, top_n=25)
fig_interactions.show()

In [0]:
print("=" * 60)
print("KEY FINDINGS")
print("=" * 60)

print(f"Model Performance:")
print(f"   Test R²: {np.mean(cv_metrics['test_r2']):.4f} ± {np.std(cv_metrics['test_r2']):.4f}")
print(f"   Test RMSE: {np.mean(cv_metrics['test_rmse']):.4f}")

print(f"Top 5 Features by SHAP Importance:")
for _, row in feature_summary.head(5).iterrows():
    direction = "↑" if row['mean_contribution'] > 0 else "↓"
    print(f"   {int(row['rank'])}. {row['feature']} ({direction} wonkiness)")

increasers = feature_summary[feature_summary['mean_contribution'] > 0].head(5)
print(f"Top features INCREASING wonkiness:")
for _, row in increasers.iterrows():
    print(f"   • {row['feature']} (+{row['mean_contribution']:.4f})")

decreasers = feature_summary[feature_summary['mean_contribution'] < 0].head(5)
print(f"Top features DECREASING wonkiness:")
for _, row in decreasers.iterrows():
    print(f"   • {row['feature']} ({row['mean_contribution']:.4f})")

print(f"\n🔗 Strongest Feature Interactions:")
for _, row in interaction_df.head(5).iterrows():
    print(f"   • {row['feature_1']} × {row['feature_2']} (strength: {row['interaction_strength']:.4f})")

In [0]:
if CONTROLL_SETTING:
    interaction_name = f"interactions_{CONTROLL_SETTING}"
    feature_summary_name = f"feature_summary_{CONTROLL_SETTING}"
    print(f"saving {interaction_name} and {feature_summary_name}")
else:
    interaction_name = "interactions"
    feature_summary_name = "feature_summary"
    print(f"saving {interaction_name} and {feature_summary_name}")


interaction_output_path = os.path.join(
    misc_dir,
    os.path.basename(paths_config['output_files'].get(interaction_name))
)
sumamry_output_path = os.path.join(
    misc_dir,
    os.path.basename(paths_config['output_files'].get(feature_summary_name))
)


feature_summary.reset_index().to_csv(sumamry_output_path, index=False)
interaction_df.reset_index().to_csv(interaction_output_path, index=False)